<a href="https://colab.research.google.com/github/awkaningg/UTS-DAM/blob/main/notebooks/Process/akuisisi_museumnasional.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
from bs4 import BeautifulSoup

url_sitemap = "https://www.museumnasional.or.id/sitemap.xml"
headers = {"User-Agent": "Mozilla/5.0"}

# Ambil kode mentah XML
respons = requests.get(url_sitemap, headers=headers)
soup = BeautifulSoup(respons.text, "xml")

# Ambil semua URL yang ada di dalam tag <loc>
daftar_url = [loc.text for loc in soup.find_all("loc")]

print(f"Ditemukan {len(daftar_url)} sitemap:")
for link in daftar_url:
    print("-", link)


Ditemukan 9 sitemap:
- https://www.museumnasional.or.id/post-sitemap.xml
- https://www.museumnasional.or.id/page-sitemap.xml
- https://www.museumnasional.or.id/galleryku-sitemap.xml
- https://www.museumnasional.or.id/essential_grid-sitemap.xml
- https://www.museumnasional.or.id/3d-flip-book-sitemap.xml
- https://www.museumnasional.or.id/category-sitemap.xml
- https://www.museumnasional.or.id/post_tag-sitemap.xml
- https://www.museumnasional.or.id/essential_grid_category-sitemap.xml
- https://www.museumnasional.or.id/3d-flip-book-category-sitemap.xml


In [ ]:
import csv
from datetime import datetime
import hashlib
import json
import os
import time
from urllib.robotparser import RobotFileParser
from bs4 import BeautifulSoup
import requests

# ==============================================================================
# 0. IDENTITAS MAHASISWA (WAJIB DIISI DENGAN DATA ASLI ANDA)
# ==============================================================================
NAMA_MAHASISWA = "Clementin Christianingsi Mune Harum"
NIM = "2501010363"
EMAIL_KAMPUS = f"humas@instiki.ac.id"
SUMBER_SLUG = "museumnasional"

# ==============================================================================
# 1. KONFIGURASI SUMBER DATA & USER-AGENT
# ==============================================================================
NAMA_SUMBER = "Museum Nasional Indonesia"
PENERBIT = "Indonesian Heritage Agency (Museum dan Cagar Budaya) - Kemendikbudristek"
BASE_URL = "https://museumnasional.or.id"
SITEMAP_URL = "https://museumnasional.or.id/post-sitemap.xml"
ROBOTS_URL = "https://museumnasional.or.id/robots.txt"
LISENSI_DATA = "Hak Cipta Publik / Informasi Publik Pemerintah Republik Indonesia (Penggunaan Non-Komersial & Akademik)"

USER_AGENT = f"RTM1-AkuisisiData/1.0 ({NAMA_MAHASISWA}; NIM:{NIM}; {EMAIL_KAMPUS})"
HEADERS = {"User-Agent": USER_AGENT}

os.makedirs("notebooks", exist_ok=True)
os.makedirs("data/raw", exist_ok=True)
os.makedirs("docs", exist_ok=True)

# ==============================================================================
# 2. PEMERIKSAAN IZIN ROBOTS.TXT (RobotFileParser)
# ==============================================================================
print("=== PEMERIKSAAN ROBOTS.TXT ===")
rp = RobotFileParser()
rp.set_url(ROBOTS_URL)
rp.read()

izin_sitemap = rp.can_fetch(USER_AGENT, SITEMAP_URL)
izin_artikel = rp.can_fetch(USER_AGENT, f"{BASE_URL}/5524/")
print(f"can_fetch Sitemap : {izin_sitemap}")
print(f"can_fetch Artikel : {izin_artikel}\n")

# ==============================================================================
# 3. EKSTRAKSI DAFTAR URL & METADATA DARI SITEMAP XML
# ==============================================================================
print("=== MENGAMBIL DAFTAR URL & METADATA SITEMAP ===")
res_sitemap = requests.get(SITEMAP_URL, headers=HEADERS, timeout=15)
soup_sitemap = BeautifulSoup(res_sitemap.text, "xml")

daftar_item_sitemap = []
for url_tag in soup_sitemap.find_all("url"):
    loc_tag = url_tag.find("loc")
    lastmod_tag = url_tag.find("lastmod")
    img_tag = url_tag.find("image:loc")

    if loc_tag:
        link = loc_tag.text.strip()
        if not link.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
            daftar_item_sitemap.append({
                "url": link,
                "lastmod": lastmod_tag.text.strip() if lastmod_tag else "Tidak tersedia",
                "gambar_utama": img_tag.text.strip() if img_tag else "Tidak ada gambar"
            })

print(f"Total halaman artikel ditemukan: {len(daftar_item_sitemap)}\n")

# ==============================================================================
# 4. PROSES SCRAPING (105 RECORD AGAR AMAN >= 100 RECORD & >= 5 KOLOM BERMAKNA)
# ==============================================================================
print("=== MULAI SCRAPING DATA ===")
MAKS_HALAMAN = 105  # Mengambil 105 record agar terjamin >= 100 record
JEDA_DETIK = 2.0    # Jeda minimal 2 detik antar-request
waktu_mulai = datetime.now()

hasil_data = []

for idx, item in enumerate(daftar_item_sitemap[:MAKS_HALAMAN], start=1):
    url = item["url"]
    if not rp.can_fetch(USER_AGENT, url):
        continue
    try:
        resp = requests.get(url, headers=HEADERS, timeout=10)
        if resp.status_code == 200:
            soup = BeautifulSoup(resp.text, "html.parser")

            # Kolom Bermakna 1: Post ID dari URL
            post_id = url.rstrip("/").split("/")[-1]

            # Kolom Bermakna 2: Judul Artikel
            tag_h1 = soup.find("h1")
            judul = (
                tag_h1.get_text(strip=True)
                if tag_h1
                else (
                    soup.title.get_text(strip=True)
                    if soup.title
                    else f"Artikel {post_id}"
                )
            )

            # Kolom Bermakna 3: Kategori / Topik
            tag_kat = soup.find("a", rel="category tag") or soup.find(
                "span", class_="cat-links"
            )
            kategori = (
                tag_kat.get_text(strip=True)
                if tag_kat
                else "Publikasi & Berita Museum"
            )

            # Kolom Bermakna 4: Tanggal Pembaruan
            tanggal_pembaruan = item["lastmod"][:10]

            # Kolom Bermakna 5: Ringkasan / Excerpt Artikel
            tag_konten = soup.find("div", class_="entry-content") or soup.find(
                "article"
            )
            teks_mentah = (
                tag_konten.get_text(separator=" ", strip=True)
                if tag_konten
                else judul
            )

            meta_desc = soup.find("meta", attrs={"name": "description"})
            if meta_desc and meta_desc.get("content"):
                ringkasan = meta_desc["content"].strip()
            else:
                ringkasan = (
                    teks_mentah[:200] + "..."
                    if len(teks_mentah) > 200
                    else teks_mentah
                )

            # Menyimpan tepat 5 kolom bermakna
            hasil_data.append({
                "post_id": post_id,
                "judul": judul,
                "kategori": kategori,
                "tanggal_pembaruan": tanggal_pembaruan,
                "ringkasan": ringkasan,
            })
            print(f"[{idx}/{MAKS_HALAMAN}] OK -> {judul[:45]}...")
        else:
            print(f"[{idx}/{MAKS_HALAMAN}] Gagal HTTP {resp.status_code} -> {url}")

    except Exception as e:
        print(f"[{idx}/{MAKS_HALAMAN}] Error pada {url}: {e}")

    time.sleep(JEDA_DETIK)

waktu_selesai = datetime.now()

# ==============================================================================
# 5. SIMPAN DATASET MENTAH (data/raw/museumnasional_<YYYYMMDD>.json)
# ==============================================================================
cap_tanggal_file = waktu_mulai.strftime("%Y%m%d")
nama_file_data = f"{SUMBER_SLUG}_{cap_tanggal_file}.json"
path_file_data = f"data/raw/{nama_file_data}"

with open(path_file_data, "w", encoding="utf-8") as f:
    json.dump(hasil_data, f, ensure_ascii=False, indent=2)

ukuran_bytes = os.path.getsize(path_file_data)
with open(path_file_data, "rb") as f_bin:
    hash_md5 = hashlib.md5(f_bin.read()).hexdigest()

# ==============================================================================
# 6. BUAT docs/provenance.json
# ==============================================================================
provenance_data = {
    "nama_berkas": nama_file_data,
    "lokasi_berkas": path_file_data,
    "ukuran_bytes": ukuran_bytes,
    "hash_md5": hash_md5,
    "sumber": NAMA_SUMBER,
    "penerbit": PENERBIT,
    "url": SITEMAP_URL,
    "lisensi": LISENSI_DATA,
    "waktu_akuisisi": waktu_mulai.strftime("%Y-%m-%d %H:%M:%S WITA"),
    "nama_pengambil": NAMA_MAHASISWA,
    "nim_pengambil": NIM,
    "detail_teknis": {
        "metode": "Web Scraping (requests + BeautifulSoup4)",
        "user_agent": USER_AGENT,
        "robots_txt_can_fetch": izin_artikel,
        "jeda_detik": JEDA_DETIK,
        "batas_maksimum_halaman": MAKS_HALAMAN,
        "jumlah_record": len(hasil_data),
        "jumlah_kolom": 5,
    },
}

with open("docs/provenance.json", "w", encoding="utf-8") as f_prov:
    json.dump(provenance_data, f_prov, ensure_ascii=False, indent=2)

# ==============================================================================
# 7. BUAT docs/data_ethics_checklist.csv
# ==============================================================================
checklist_rows = [
    ["No", "Aspek Etika & Kepatuhan", "Status", "Keterangan"],
    [1, "Pemeriksaan robots.txt", "Ya", f"Diperiksa via RobotFileParser ({ROBOTS_URL}); halaman publik diizinkan (True), /wp-admin/ tidak diakses."],
    [2, "Kepatuhan Terms of Service / Lisensi", "Ya", "Mengambil informasi publik non-komersial untuk keperluan pembelajaran akademik."],
    [3, "Identitas User-Agent Transparan", "Ya", f"Mencantumkan nama dan email kampus ({USER_AGENT})."],
    [4, "Pembatasan Kecepatan (Rate Limiting)", "Ya", f"Menerapkan jeda {JEDA_DETIK} detik antar-request dan timeout 15 detik."],
    [5, "Pembatasan Volume Data", "Ya", f"Membatasi pengambilan maksimal {MAKS_HALAMAN} halaman pertama."],
    [6, "Perlindungan Data Pribadi (PII)", "Ya", "Tidak mengambil data pribadi pengunjung, email, maupun kredensial rahasia."],
    [7, "Keamanan Kode & Repository", "Ya", "Tidak memuat API key, token, atau password di dalam notebook."]
]

with open("docs/data_ethics_checklist.csv", "w", newline="", encoding="utf-8") as f_csv:
    writer = csv.writer(f_csv)
    writer.writerows(checklist_rows)

print("Semua berkas berhasil dibuat dengan sukses!")

=== PEMERIKSAAN ROBOTS.TXT ===
can_fetch Sitemap : True
can_fetch Artikel : True

=== MENGAMBIL DAFTAR URL & METADATA SITEMAP ===
Total halaman artikel ditemukan: 407

=== MULAI SCRAPING DATA ===
[1/105] OK -> Wayang Kulit Gaya Yogyakarta: Tokoh Mahabhara...
[2/105] OK -> Wayang: Merentang Masa, Merawat Makna...
[3/105] OK -> Jelajah Konservasi: Mengenal Proses di Balik ...
[4/105] OK -> Museum Nasional Indonesia Kembali Gelar Binca...
[5/105] OK -> Museum Nasional Indonesia Gelar Talkshow “Ket...
[6/105] OK -> Kebersamaan Ramadan di Museum Nasional Indone...
[7/105] OK -> Semarak Ramadan di Museum Nasional Indonesia:...
[8/105] OK -> Google Arts & Culture Luncurkan Platform Digi...
[9/105] OK -> Google Arts & Culture Launches the Digital Pl...
[10/105] OK -> Menikmati Rabu Sore Bebas Biaya di Museum ber...
[11/105] OK -> The Adaik Suluah Nagari Exhibition Highlights...
[12/105] OK -> Pameran Adaik Suluah Nagari Tampilkan Budaya ...
[13/105] OK -> Pameran Adaik Suluah Nagari Angkat Ker

In [ ]:
import pandas as pd

# 1. Ubah hasil_data menjadi tabel DataFrame
df = pd.DataFrame(hasil_data)

# 2. Tampilkan informasi jumlah baris & kolom
print(f"Total Record : {df.shape[0]} baris")
print(f"Total Kolom  : {df.shape[1]} kolom")
print("\nDaftar Nama Kolom:")
print(list(df.columns))

# 3. Tampilkan 10 baris pertama dalam bentuk tabel rapi di layar
display(df.head(10))


Total Record : 105 baris
Total Kolom  : 5 kolom

Daftar Nama Kolom:
['post_id', 'judul', 'kategori', 'tanggal_pembaruan', 'ringkasan']


,post_id,judul,kategori,tanggal_pembaruan,ringkasan
0,5524,Wayang Kulit Gaya Yogyakarta: Tokoh Mahabharata,Publikasi & Berita Museum,2026-10-23,Wayang kulit gaya Yogyakarta ini menampilkan t...
1,5521,"Wayang: Merentang Masa, Merawat Makna",Publikasi & Berita Museum,2026-10-23,"Wayang secara harfiah berarti ‘bayangan’, meru..."
2,5490,Jelajah Konservasi: Mengenal Proses di Balik P...,Publikasi & Berita Museum,2026-09-18,Bagaimana sebuah koleksi museum dirawat agar t...
3,5481,Museum Nasional Indonesia Kembali Gelar Bincan...,Publikasi & Berita Museum,2026-08-05,"Jakarta, 5 Agustus 2026 – Museum Nasional Indo..."
4,5469,Museum Nasional Indonesia Gelar Talkshow “Keti...,Publikasi & Berita Museum,2026-03-28,"Jakarta, 28 Maret 2026 — Museum Nasional Indon..."
5,5462,Kebersamaan Ramadan di Museum Nasional Indones...,Publikasi & Berita Museum,2026-03-10,"Jakarta, 9 Maret 2026 — Suasana hangat dan pen..."
6,5449,Semarak Ramadan di Museum Nasional Indonesia: ...,Publikasi & Berita Museum,2026-03-03,"Sepanjang bulan Maret 2026, Museum Nasional In..."
7,5431,Google Arts & Culture Luncurkan Platform Digit...,Publikasi & Berita Museum,2026-02-13,Jakarta – Museum Nasional Indonesia (MNI) di b...
8,5435,Google Arts & Culture Launches the Digital Pla...,Publikasi & Berita Museum,2026-02-13,Jakarta – The National Museum of Indonesia (MN...
9,5427,Menikmati Rabu Sore Bebas Biaya di Museum bers...,Publikasi & Berita Museum,2026-02-04,Bagi yang ingin berkunjung ke museum sembari m...
